# U-Net like CNN with skip-connections (TEST)

*For color diversity research*

## Config

In [ ]:
CLEARML_SAVE_TASK = True

In [ ]:
CONFIG = {
    "batch_size": 32,
    "num_workers": 6,
    "prefetch_factor": 4,
    "model_version": "v2",
}

## Imports
### Libs

In [ ]:
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from clearml import Dataset, Task, TaskTypes
from torch.utils.data import DataLoader

### Chromatica modules

In [ ]:
from chromatica.charts import generic
from chromatica.datasets.dataset import ImageDataset
from chromatica.nn import load_cnn

In [ ]:
CNN = load_cnn(CONFIG["model_version"])

### Check for CUDA or MPS

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS is used")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is used")
else:
    device = torch.device("cpu")
    print("CPU is used")

### Useful functions

In [ ]:
def mps_autocast_or_null():
    if device.type == "mps":
        try:
            return torch.autocast("mps", dtype=torch.float16)
        except Exception:
            return nullcontext()
    return nullcontext()

In [ ]:
def loader(dataset):
    return DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        num_workers=CONFIG["num_workers"],
        prefetch_factor=CONFIG["prefetch_factor"],
        persistent_workers=True,
        pin_memory=(device == torch.device("cuda")),
    )

## Test task init

In [ ]:
task = Task.init(
    project_name="Chromatica",
    task_name="Test CNN for analyze diversity of colors",
    task_type=TaskTypes.testing,
    tags=[
        CONFIG["model_version"],
    ],
)

In [ ]:
logger = task.get_logger()

In [ ]:
CONFIG = task.connect_configuration(CONFIG)

In [ ]:
train_task = Task.get_task(
    project_name="Chromatica",
    task_name="Train CNN for analyze diversity of colors",
    tags=[
        CONFIG["model_version"],
    ],
)

## Test
### Only `Food101`
#### Load model

In [ ]:
model_food101_artifact = train_task.artifacts["model_food101"].get_local_copy()

In [ ]:
model_food101 = CNN()
model_food101.load_state_dict(torch.load(model_food101_artifact))
model_food101 = model_food101.to(device)

#### Load dataset

In [ ]:
food101_path = Path(
    Dataset.get(dataset_project="Colorization", dataset_name="Food101").get_local_copy()
)

In [ ]:
food101 = ImageDataset(food101_path / "test")

#### Testing

In [ ]:
%%time

dataset_name = "Food101"
title_loss = f"{dataset_name} Loss"

model_food101.eval()

sum_sq_err = 0.0
n_elems = 0

with torch.inference_mode():
    for x_, y_, _ in loader(food101):
        x = x_.to(device, non_blocking=True)
        y = y_.to(device, non_blocking=True)

        with mps_autocast_or_null():
            pred = model_food101(x)

        diff2_sum = (pred.float() - y.float()).pow(2).sum().item()
        sum_sq_err += diff2_sum
        n_elems += y.numel()

mse = sum_sq_err / max(1, n_elems)

logger.report_single_value("test_mse_food101", float(mse))

print(f"[{dataset_name}] Test MSE: {mse:.6f}\n")

#### Visualization

In [ ]:
idx = 0

x, y, _ = food101[idx]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(generic.lab2image(x, y))

axes[0].axis("off")
axes[0].set_title("Original")

pred = model_food101(x[None, :, :, :].to(device))
axes[1].imshow(generic.lab2image(x, pred[0].cpu()))
axes[1].axis("off")
axes[1].set_title("Predict")

plt.tight_layout()

logger.report_matplotlib_figure(
    title=f"Predict (idx={idx})",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

plt.show()

In [ ]:
idx = 1500

x, y, _ = food101[idx]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(generic.lab2image(x, y))

axes[0].axis("off")
axes[0].set_title("Original")

pred = model_food101(x[None, :, :, :].to(device))
axes[1].imshow(generic.lab2image(x, pred[0].cpu()))
axes[1].axis("off")
axes[1].set_title("Predict")

plt.tight_layout()

logger.report_matplotlib_figure(
    title=f"Predict (idx={idx})",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

plt.show()

In [ ]:
idx = 4500

x, y, _ = food101[idx]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(generic.lab2image(x, y))

axes[0].axis("off")
axes[0].set_title("Original")

pred = model_food101(x[None, :, :, :].to(device))
axes[1].imshow(generic.lab2image(x, pred[0].cpu()))
axes[1].axis("off")
axes[1].set_title("Predict")

plt.tight_layout()

logger.report_matplotlib_figure(
    title=f"Predict (idx={idx})",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

plt.show()

## Test
### Only `COCO`
#### Load model

In [ ]:
model_coco_artifact = train_task.artifacts["model_coco"].get_local_copy()

In [ ]:
model_coco = CNN()
model_coco.load_state_dict(torch.load(model_coco_artifact))
model_coco = model_coco.to(device)

#### Load dataset

In [ ]:
coco_path = Path(
    Dataset.get(dataset_project="Colorization", dataset_name="COCO").get_local_copy()
)

In [ ]:
coco = ImageDataset(coco_path / "test")

#### Testing

In [ ]:
%%time

dataset_name = "COCO"
title_loss = f"{dataset_name} Loss"

model_coco.eval()

sum_sq_err = 0.0
n_elems = 0

with torch.inference_mode():
    for x_, y_, _ in loader(coco):
        x = x_.to(device, non_blocking=True)
        y = y_.to(device, non_blocking=True)

        with mps_autocast_or_null():
            pred = model_coco(x)

        diff2_sum = (pred.float() - y.float()).pow(2).sum().item()
        sum_sq_err += diff2_sum
        n_elems += y.numel()

mse = sum_sq_err / max(1, n_elems)

logger.report_single_value("test_mse_coco", float(mse))

print(f"[{dataset_name}] Test MSE: {mse:.6f}\n")

#### Visualization

In [ ]:
idx = 0

x, y, _ = coco[idx]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(generic.lab2image(x, y))

axes[0].axis("off")
axes[0].set_title("Original")

pred = model_coco(x[None, :, :, :].to(device))
axes[1].imshow(generic.lab2image(x, pred[0].cpu()))
axes[1].axis("off")
axes[1].set_title("Predict")

plt.tight_layout()

logger.report_matplotlib_figure(
    title=f"Predict (idx={idx})",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

plt.show()

In [ ]:
idx = 1500

x, y, _ = coco[idx]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(generic.lab2image(x, y))

axes[0].axis("off")
axes[0].set_title("Original")

pred = model_coco(x[None, :, :, :].to(device))
axes[1].imshow(generic.lab2image(x, pred[0].cpu()))
axes[1].axis("off")
axes[1].set_title("Predict")

plt.tight_layout()

logger.report_matplotlib_figure(
    title=f"Predict (idx={idx})",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

plt.show()

In [ ]:
idx = 4500

x, y, _ = coco[idx]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(generic.lab2image(x, y))

axes[0].axis("off")
axes[0].set_title("Original")

pred = model_coco(x[None, :, :, :].to(device))
axes[1].imshow(generic.lab2image(x, pred[0].cpu()))
axes[1].axis("off")
axes[1].set_title("Predict")

plt.tight_layout()

logger.report_matplotlib_figure(
    title=f"Predict (idx={idx})",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

plt.show()

In [ ]:
if CLEARML_SAVE_TASK:
    task.mark_completed()
else:
    task.close()
    task.set_archived(True)